# 분류 결과를 무엇과 비교하는가?

학습 영역과 검증 영역을 분리하고 기준자료의 한계 이해

**할 일** · 서쪽 영역 학습 → 동쪽 영역 검증, 오차행렬 해석

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B2', 'B3', 'B4', 'B8', 'B11'])
            .multiply(0.0001).updateMask(clear)
            .copyProperties(image, ['system:time_start']))
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi).filterDate('2020-05-01', '2020-10-01')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2))

image = collection.median().clip(roi)
bands = ['B2','B3','B4','B8','B11']
wc = ee.ImageCollection('ESA/WorldCover/v100').first().select('Map')
classes = [10,20,30,40,50,60,70,80,90,95,100]
labels = wc.remap(classes, list(range(11))).rename('label').toByte()
stack = image.select(bands).addBands(labels)
west = ee.Geometry.Rectangle([126.80,37.43,126.94,37.70], geodesic=False)
east = ee.Geometry.Rectangle([127.01,37.43,127.15,37.70], geodesic=False)
training = stack.stratifiedSample(numPoints=60,classBand='label',region=west,scale=30,seed=7,geometries=False)
validation = stack.stratifiedSample(numPoints=40,classBand='label',region=east,scale=30,seed=8,geometries=False)
model = ee.Classifier.smileRandomForest(numberOfTrees=50,seed=7).train(training,'label',bands)
classified = image.select(bands).classify(model)
matrix = validation.classify(model).errorMatrix('label','classification',ee.List.sequence(0,10))

In [ ]:
print('train / validation:', training.size().getInfo(), validation.size().getInfo())
print('confusion matrix:', matrix.getInfo())
print('class-stratified sample agreement:', matrix.accuracy().getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(classified.randomVisualizer(), {}, 'RF classes')
Map.addLayer(west, {'color':'blue'}, 'Training region')
Map.addLayer(east, {'color':'red'}, 'Validation region')
Map

학습 영역과 검증 영역을 지도에서 확인하시오. 혼동이 큰 범주를 찾아 RGB와 비교하시오.

**해석 범위** · WorldCover는 동일 계열 위성자료로 만든 기준 지도. 독립 현장 정답이 아니므로 수치는 ‘기준 지도와의 일치도’. 범주별 층화 표본이므로 단순 전체 일치도는 지역 면적 가중 정확도와 다름. 공간 분리만으로 모든 의존성 제거를 보장하지 않음

## 자료와 해석 범위

- [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)

- [WorldCover 2020](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v100)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단